# 05 — Inference Demo

This notebook loads the saved model and predicts emotions for custom text.

Use this notebook to test whether the trained model works before connecting it to a web app, chatbot, or backend.

In [2]:
# Import required libraries
from pathlib import Path
import json

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\Taruni\Desktop\My Projects\Pet_Pal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Set paths and load model

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models" / "emotion_model_v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Model directory:", MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("Model loaded successfully.")

Using device: cpu
Model directory: c:\Users\Taruni\Desktop\PetChat-2.0\emotion_classifier\models\emotion_model_v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6032.49it/s]

Model loaded successfully.


## 2. Create prediction function

In [4]:
def predict_emotion(text, return_scores=True):
    # Predict emotion from text.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(torch.argmax(probabilities).item())
    predicted_label = model.config.id2label[predicted_id]
    confidence = float(probabilities[predicted_id].item())

    result = {
        "text": text,
        "predicted_emotion": predicted_label,
        "confidence": round(confidence, 4)
    }

    if return_scores:
        all_scores = {
            model.config.id2label[i]: round(float(probabilities[i].item()), 4)
            for i in range(len(probabilities))
        }
        result["all_scores"] = all_scores

    return result

## 3. Test with sample sentences

In [5]:
sample_texts = [
    "I am so happy and excited today!",
    "Thank you so much, I really appreciate your help.",
    "I feel very sad and disappointed.",
    "I am angry because this is not fair.",
    "I am scared about what will happen next.",
    "I feel nervous and stressed about my exam.",
    "I do not understand what is going on here.",
    "Okay, that sounds fine to me."
]

results = []

for text in sample_texts:
    prediction = predict_emotion(text, return_scores=False)
    results.append(prediction)

results_df = pd.DataFrame(results)
display(results_df)

,text,predicted_emotion,confidence
0,I am so happy and excited today!,happy,0.8742
1,"Thank you so much, I really appreciate your help.",happy,0.9206
2,I feel very sad and disappointed.,sad,0.8032
3,I am angry because this is not fair.,sad,0.2805
4,I am scared about what will happen next.,anxious,0.9364
5,I feel nervous and stressed about my exam.,stressed,0.8549
6,I do not understand what is going on here.,confused,0.4041
7,"Okay, that sounds fine to me.",happy,0.6291


## 4. Check full probability scores for one sentence

In [6]:
text = "I am worried about my results and I cannot relax."

prediction = predict_emotion(text, return_scores=True)

print("Text:", prediction["text"])
print("Predicted emotion:", prediction["predicted_emotion"])
print("Confidence:", prediction["confidence"])

display(pd.DataFrame(
    prediction["all_scores"].items(),
    columns=["emotion", "probability"]
).sort_values("probability", ascending=False))

Text: I am worried about my results and I cannot relax.
Predicted emotion: stressed
Confidence: 0.7432


,emotion,probability
5,stressed,0.7432
4,anxious,0.2031
2,sad,0.0244
1,calm,0.0093
0,happy,0.0072
3,angry,0.0065
6,confused,0.0064


## 5. Try your own sentence

In [7]:
# Change this sentence and run the cell again.
my_text = "Im enjoying my day today "

predict_emotion(my_text, return_scores=True)

{'text': 'Im enjoying my day today ',
 'predicted_emotion': 'happy',
 'confidence': 0.8227,
 'all_scores': {'happy': 0.8227,
  'calm': 0.099,
  'sad': 0.025,
  'angry': 0.0117,
  'anxious': 0.0048,
  'stressed': 0.0101,
  'confused': 0.0267}}

## Inference Summary

This notebook confirms whether the saved model can classify new text into the 7 project emotions.

For a final app, you can convert the prediction function into a small Python file such as:

```text
app/classifier.py
```